# 🌽 Safrinha 2025 — criação de amostras, retreinamento e validação

Este notebook agora começa pelo passo correto: **criar/exportar um TFRecord específico da safrinha de 2025** antes do retreinamento. Isso evita que o treinamento use, por engano, as amostras da safra principal.

Fluxo recomendado:

1. Execute o **Passo 1** para gerar `AMOSTRAS_SAFRINHA_2025_*.tfrecord.gz` no Google Drive.
2. Aguarde a tarefa de exportação do Earth Engine terminar.
3. Execute o **Passo 2** para treinar a U-Net usando somente o TFRecord da safrinha.
4. Execute o **Passo 3** para fazer a prova visual.


## Passo 1 — Criar as amostras TFRecord da safrinha 2025

Preencha os assets/geométricas abaixo e execute a célula. Ela cria composições Sentinel-2 para duas janelas da safrinha, rasteriza o gabarito de pivôs e exporta patches 129×129 em TFRecord para o Google Drive.

O notebook já vem configurado com `EE_PROJECT_ID = 'sefazgogeoprocessamento'` e com o gabarito `projects/sefazgogeoprocessamento/assets/Gabarito_Milho_UPLOAD`. Como a área é o **município todo**, o notebook agora tenta buscar automaticamente o limite de `Jussara-GO` na coleção pública `FAO/GAUL/2015/level2`; se quiser usar um limite próprio, preencha `ASSET_AREA_ESTUDO` ou `AREA_ESTUDO_GEOMETRY`.

Para o município inteiro, deixe `MODO_TESTE_RAPIDO = True` na primeira rodada; isso exporta menos patches e divide a exportação em partes menores, evitando o erro `Computed value is too large` antes da exportação final.

> Importante: depois de rodar a célula, acompanhe a tarefa no painel **Tasks** do Earth Engine/Colab e espere finalizar antes de treinar.

> Observação: foi criada a versão `PATCHES_V2` porque os TFRecords antigos podiam salvar `label_chip` como valor escalar, causando erro de reshape no TensorFlow.


In [ ]:
# Se estiver no Colab e ainda não tiver geemap, descomente a linha abaixo.
# !pip -q install geemap

import ee
import os
from google.colab import drive

# 1. Montar Drive e autenticar Earth Engine
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Informe aqui o ID do projeto Google Cloud vinculado/habilitado no Earth Engine.
# Exemplo: EE_PROJECT_ID = 'meu-projeto-earthengine'
# Dica: no Google Cloud Console, copie o campo "Project ID" (não é o nome amigável do projeto).
EE_PROJECT_ID = 'sefazgogeoprocessamento'

def inicializar_earth_engine(project_id):
    project_id = project_id or os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('GCLOUD_PROJECT')
    ee.Authenticate()

    if not project_id:
        raise ValueError(
            'Defina EE_PROJECT_ID com o Project ID do Google Cloud habilitado no Earth Engine antes de executar. '
            "Exemplo: EE_PROJECT_ID = 'meu-projeto-earthengine'. "
            'Sem esse valor, o Colab pode retornar: ee.Initialize: no project found.'
        )

    ee.Initialize(project=project_id)
    print(f'✅ Earth Engine inicializado com o projeto: {project_id}')

inicializar_earth_engine(EE_PROJECT_ID)

# --- CONFIGURAÇÕES DA EXPORTAÇÃO DA SAFRINHA 2025 ---
# Pasta do Google Drive onde o TFRecord será salvo.
DRIVE_FOLDER = 'Tese_IA_Jussara'

# Nome base do arquivo exportado. O treinamento busca por AMOSTRAS_SAFRINHA_2025.
NOME_EXPORTACAO = 'AMOSTRAS_SAFRINHA_2025_JUSSARA_PATCHES_V2'
VERSAO_AMOSTRAS = 'PATCHES_V2'

# Área de estudo: como a área é o município todo, informe o limite municipal aqui.
# Exemplo asset: 'projects/sefazgogeoprocessamento/assets/Limite_Municipio_Jussara'
# Se não tiver asset, cole a geometria GeoJSON em AREA_ESTUDO_GEOMETRY.
ASSET_AREA_ESTUDO = None
AREA_ESTUDO_GEOMETRY = None
AREA_E_MUNICIPIO_INTEIRO = True

# Limite municipal padrão quando ASSET_AREA_ESTUDO/AREA_ESTUDO_GEOMETRY ficarem vazios.
# Usa uma coleção pública do GEE para buscar Jussara-GO automaticamente.
MUNICIPIO_NOME = 'Jussara'
ESTADO_NOME = 'Goias'
COLECAO_MUNICIPIOS_GEE = 'FAO/GAUL/2015/level2'

# Asset com os pivôs/gabarito da safrinha 2025.
# Pode ser FeatureCollection de polígonos ou Image raster 0/1.
ASSET_LABEL_SAFRINHA_2025 = 'projects/sefazgogeoprocessamento/assets/Gabarito_Milho_UPLOAD'
LABEL_EH_VETOR = True

# Janelas temporais da safrinha 2025. Ajuste se sua janela agronômica for diferente.
JANELA_1_INICIO = '2025-02-01'
JANELA_1_FIM = '2025-04-30'
JANELA_2_INICIO = '2025-05-01'
JANELA_2_FIM = '2025-07-31'

# Parâmetros compatíveis com o notebook de treinamento.
# Para município inteiro, comece com MODO_TESTE_RAPIDO=True para não esperar horas no GEE.
MODO_TESTE_RAPIDO = True
SCALE = 10
READ_SIZE = 129
KERNEL_RADIUS = READ_SIZE // 2
AMOSTRAS_POSITIVAS = 300 if MODO_TESTE_RAPIDO else 2500
AMOSTRAS_NEGATIVAS = 300 if MODO_TESTE_RAPIDO else 2500
EXPORT_TILE_SCALE = 8
# Divide a exportação em partes menores para evitar erro 'Computed value is too large' no GEE.
NUM_PARTES_EXPORT = 12 if MODO_TESTE_RAPIDO else 50
SEED = 2025

if not ASSET_LABEL_SAFRINHA_2025:
    raise ValueError('Defina ASSET_LABEL_SAFRINHA_2025 com o gabarito da safrinha 2025 antes de exportar.')

if LABEL_EH_VETOR:
    pivos = ee.FeatureCollection(ASSET_LABEL_SAFRINHA_2025)
else:
    label_asset = ee.Image(ASSET_LABEL_SAFRINHA_2025)

def obter_area_municipio_padrao():
    municipios = ee.FeatureCollection(COLECAO_MUNICIPIOS_GEE)
    candidatos = municipios.filter(
        ee.Filter.And(
            ee.Filter.eq('ADM0_NAME', 'Brazil'),
            ee.Filter.eq('ADM1_NAME', ESTADO_NOME),
            ee.Filter.eq('ADM2_NAME', MUNICIPIO_NOME),
        )
    )
    total = candidatos.size().getInfo()
    if total == 0:
        raise ValueError(
            f'Não encontrei {MUNICIPIO_NOME}-{ESTADO_NOME} em {COLECAO_MUNICIPIOS_GEE}. '
            'Informe ASSET_AREA_ESTUDO ou AREA_ESTUDO_GEOMETRY manualmente.'
        )
    print(f'✅ Área de estudo definida automaticamente: município de {MUNICIPIO_NOME}-{ESTADO_NOME}.')
    return candidatos.geometry()

if ASSET_AREA_ESTUDO:
    area_estudo = ee.FeatureCollection(ASSET_AREA_ESTUDO).geometry()
elif AREA_ESTUDO_GEOMETRY:
    area_estudo = ee.Geometry(AREA_ESTUDO_GEOMETRY)
elif AREA_E_MUNICIPIO_INTEIRO:
    area_estudo = obter_area_municipio_padrao()
elif LABEL_EH_VETOR:
    area_estudo = pivos.geometry().bounds()
    print('ℹ️ ASSET_AREA_ESTUDO não foi informado; usando a extensão do gabarito como área de estudo.')
else:
    area_estudo = label_asset.geometry().bounds()
    print('ℹ️ ASSET_AREA_ESTUDO não foi informado; usando a extensão da imagem de gabarito como área de estudo.')

def mascarar_sentinel2_sr(img):
    scl = img.select('SCL')
    mascara = (
        scl.neq(3)   # sombra de nuvem
        .And(scl.neq(8))   # nuvem média probabilidade
        .And(scl.neq(9))   # nuvem alta probabilidade
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # neve/gelo
    )
    return img.updateMask(mascara).divide(10000).copyProperties(img, ['system:time_start'])

def composicao_s2(inicio, fim, sufixo):
    colecao = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(area_estudo)
        .filterDate(inicio, fim)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
        .map(mascarar_sentinel2_sr)
    )
    base = colecao.median().clip(area_estudo)
    ndvi = base.normalizedDifference(['B8', 'B4']).rename(f'NDVI_{sufixo}')
    return base.select(['B4', 'B8'], [f'R_{sufixo}', f'NIR_{sufixo}']).addBands(ndvi)

img_1 = composicao_s2(JANELA_1_INICIO, JANELA_1_FIM, '1')
img_2 = composicao_s2(JANELA_2_INICIO, JANELA_2_FIM, '2')

if LABEL_EH_VETOR:
    pivos = pivos.filterBounds(area_estudo)
    label_chip = ee.Image(0).byte().paint(pivos, 1).rename('label_chip').clip(area_estudo)
else:
    label_chip = label_asset.gt(0).byte().rename('label_chip').clip(area_estudo)

imagem_base = img_1.addBands(img_2).addBands(label_chip).unmask(0).float()
print(f'⚙️ MODO_TESTE_RAPIDO={MODO_TESTE_RAPIDO} | positivas={AMOSTRAS_POSITIVAS} | negativas={AMOSTRAS_NEGATIVAS} | tileScale={EXPORT_TILE_SCALE}')

# Pontos balanceados para não depender das amostras da safra principal.
# Importante: usamos uma banda temporária ('classe_amostra') para a amostragem.
# Se a propriedade do ponto também se chamar 'label_chip', ela sobrescreve o patch 129x129
# exportado por sampleRegions e causa erro de reshape no TensorFlow.
classe_amostra = label_chip.rename('classe_amostra')
pontos = classe_amostra.stratifiedSample(
    numPoints=AMOSTRAS_POSITIVAS + AMOSTRAS_NEGATIVAS,
    classBand='classe_amostra',
    region=area_estudo,
    scale=SCALE,
    classValues=[0, 1],
    classPoints=[AMOSTRAS_NEGATIVAS, AMOSTRAS_POSITIVAS],
    seed=SEED,
    geometries=True,
    tileScale=EXPORT_TILE_SCALE,
)

# Converte cada ponto em um patch 129x129 por banda, igual ao READ_SIZE esperado no treinamento.
# A exportação é dividida em partes para evitar o erro 'Computed value is too large'.
kernel = ee.Kernel.square(radius=KERNEL_RADIUS, units='pixels', normalize=False)
pontos_com_shard = pontos.randomColumn('shard_random', seed=SEED)

print('📦 Exportando TFRecords específicos da safrinha 2025 para o Google Drive...')
print(f'Nome base esperado: {NOME_EXPORTACAO}')
print(f'🧩 Partes de exportação: {NUM_PARTES_EXPORT}')

tasks = []
for parte in range(NUM_PARTES_EXPORT):
    inicio = parte / NUM_PARTES_EXPORT
    fim = (parte + 1) / NUM_PARTES_EXPORT
    pontos_parte = pontos_com_shard.filter(
        ee.Filter.And(
            ee.Filter.gte('shard_random', inicio),
            ee.Filter.lt('shard_random', fim),
        )
    )
    patches = imagem_base.neighborhoodToArray(kernel).sampleRegions(
        collection=pontos_parte,
        scale=SCALE,
        properties=[],
        geometries=False,
        tileScale=EXPORT_TILE_SCALE,
    )
    sufixo = f'PARTE_{parte + 1:02d}_DE_{NUM_PARTES_EXPORT:02d}'
    nome_parte = f'{NOME_EXPORTACAO}_{sufixo}'
    task = ee.batch.Export.table.toDrive(
        collection=patches,
        description=nome_parte,
        folder=DRIVE_FOLDER,
        fileNamePrefix=nome_parte,
        fileFormat='TFRecord',
        selectors=['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2', 'label_chip'],
    )
    task.start()
    tasks.append(task)
    print(f'✅ Tarefa iniciada: {nome_parte} | Task ID: {task.id}')

print('✅ Todas as partes foram iniciadas. Aguarde finalizar no painel Tasks antes de executar o treinamento.')


## Passo 2 — Retreinar usando somente as amostras da safrinha 2025

Esta etapa **não usa fallback genérico para amostras da safra**. Ela procura apenas TFRecords criados para a safrinha 2025 ou o caminho manual informado em `CAMINHO_TFRECORD_SAFRINHA`. Para município inteiro, a exportação pode ser demorada; por isso o notebook começa em `MODO_TESTE_RAPIDO = True` com menos amostras e exportação dividida em partes menores.

Se o TFRecord ainda não existir, a própria célula inicia automaticamente a exportação da safrinha no Earth Engine e tenta aguardar a task finalizar para continuar o treino na mesma execução. Se o arquivo ainda não aparecer no Drive, ela pausa sem traceback e orienta executar novamente depois.

O treino procura por arquivos `PATCHES_V2`, ignorando exports antigos com `label_chip` escalar.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
import time
from google.colab import drive

# 1. Montar Drive (Obrigatório no Colab)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURAÇÕES DA SAFRINHA 2025 ---
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
NOME_CENARIO = 'Safrinha_2025'

# Se quiser forçar um arquivo específico, cole o caminho completo aqui.
# Exemplo: CAMINHO_TFRECORD_SAFRINHA = '/content/drive/MyDrive/Tese_IA_Jussara/AMOSTRAS_SAFRINHA_2025_JUSSARA.tfrecord.gz'
CAMINHO_TFRECORD_SAFRINHA = None

# Se não encontrar o TFRecord, a célula pode criar automaticamente a tarefa de exportação.
AUTO_CRIAR_TFRECORD_SE_AUSENTE = True

# Se criar a exportação automaticamente, aguarda a task terminar e tenta continuar o treino na mesma execução.
AGUARDAR_EXPORTACAO_E_TREINAR = True
TEMPO_MAX_ESPERA_EXPORTACAO_MIN = 180
INTERVALO_CHECAGEM_EXPORTACAO_SEG = 60

# Configurações usadas para criar o TFRecord automaticamente quando ele ainda não existe.
EE_PROJECT_ID = 'sefazgogeoprocessamento'
DRIVE_FOLDER = 'Tese_IA_Jussara'
NOME_EXPORTACAO = 'AMOSTRAS_SAFRINHA_2025_JUSSARA_PATCHES_V2'
VERSAO_AMOSTRAS = 'PATCHES_V2'
USAR_APENAS_VERSAO_ATUAL = True
# Para município inteiro, informe o limite municipal em ASSET_AREA_ESTUDO ou AREA_ESTUDO_GEOMETRY.
ASSET_AREA_ESTUDO = None
AREA_ESTUDO_GEOMETRY = None
AREA_E_MUNICIPIO_INTEIRO = True
MUNICIPIO_NOME = 'Jussara'
ESTADO_NOME = 'Goias'
COLECAO_MUNICIPIOS_GEE = 'FAO/GAUL/2015/level2'
ASSET_LABEL_SAFRINHA_2025 = 'projects/sefazgogeoprocessamento/assets/Gabarito_Milho_UPLOAD'
LABEL_EH_VETOR = True
JANELA_1_INICIO = '2025-02-01'
JANELA_1_FIM = '2025-04-30'
JANELA_2_INICIO = '2025-05-01'
JANELA_2_FIM = '2025-07-31'
MODO_TESTE_RAPIDO = True
SCALE = 10
READ_SIZE_EXPORT = 129
KERNEL_RADIUS = READ_SIZE_EXPORT // 2
AMOSTRAS_POSITIVAS = 300 if MODO_TESTE_RAPIDO else 2500
AMOSTRAS_NEGATIVAS = 300 if MODO_TESTE_RAPIDO else 2500
EXPORT_TILE_SCALE = 8
# Divide a exportação em partes menores para evitar erro 'Computed value is too large' no GEE.
NUM_PARTES_EXPORT = 12 if MODO_TESTE_RAPIDO else 50
SEED = 2025

def iniciar_exportacao_tfrecord_safrinha():
    import ee

    project_id = EE_PROJECT_ID or os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('GCLOUD_PROJECT')
    if not project_id:
        raise ValueError('Defina EE_PROJECT_ID antes de criar automaticamente o TFRecord da safrinha 2025.')

    ee.Authenticate()
    ee.Initialize(project=project_id)
    print(f'✅ Earth Engine inicializado com o projeto: {project_id}')

    if not ASSET_LABEL_SAFRINHA_2025:
        raise ValueError('Defina ASSET_LABEL_SAFRINHA_2025 antes de criar automaticamente o TFRecord.')

    if LABEL_EH_VETOR:
        pivos = ee.FeatureCollection(ASSET_LABEL_SAFRINHA_2025)
    else:
        label_asset = ee.Image(ASSET_LABEL_SAFRINHA_2025)

    def obter_area_municipio_padrao():
        municipios = ee.FeatureCollection(COLECAO_MUNICIPIOS_GEE)
        candidatos = municipios.filter(
            ee.Filter.And(
                ee.Filter.eq('ADM0_NAME', 'Brazil'),
                ee.Filter.eq('ADM1_NAME', ESTADO_NOME),
                ee.Filter.eq('ADM2_NAME', MUNICIPIO_NOME),
            )
        )
        total = candidatos.size().getInfo()
        if total == 0:
            raise ValueError(
                f'Não encontrei {MUNICIPIO_NOME}-{ESTADO_NOME} em {COLECAO_MUNICIPIOS_GEE}. '
                'Informe ASSET_AREA_ESTUDO ou AREA_ESTUDO_GEOMETRY manualmente.'
            )
        print(f'✅ Área de estudo definida automaticamente: município de {MUNICIPIO_NOME}-{ESTADO_NOME}.')
        return candidatos.geometry()

    if ASSET_AREA_ESTUDO:
        area_estudo = ee.FeatureCollection(ASSET_AREA_ESTUDO).geometry()
    elif AREA_ESTUDO_GEOMETRY:
        area_estudo = ee.Geometry(AREA_ESTUDO_GEOMETRY)
    elif AREA_E_MUNICIPIO_INTEIRO:
        area_estudo = obter_area_municipio_padrao()
    elif LABEL_EH_VETOR:
        area_estudo = pivos.geometry().bounds()
    else:
        area_estudo = label_asset.geometry().bounds()

    def mascarar_sentinel2_sr(img):
        scl = img.select('SCL')
        mascara = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        return img.updateMask(mascara).divide(10000).copyProperties(img, ['system:time_start'])

    def composicao_s2(inicio, fim, sufixo):
        colecao = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(area_estudo)
            .filterDate(inicio, fim)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
            .map(mascarar_sentinel2_sr)
        )
        base = colecao.median().clip(area_estudo)
        ndvi = base.normalizedDifference(['B8', 'B4']).rename(f'NDVI_{sufixo}')
        return base.select(['B4', 'B8'], [f'R_{sufixo}', f'NIR_{sufixo}']).addBands(ndvi)

    img_1 = composicao_s2(JANELA_1_INICIO, JANELA_1_FIM, '1')
    img_2 = composicao_s2(JANELA_2_INICIO, JANELA_2_FIM, '2')

    if LABEL_EH_VETOR:
        pivos = pivos.filterBounds(area_estudo)
        label_chip = ee.Image(0).byte().paint(pivos, 1).rename('label_chip').clip(area_estudo)
    else:
        label_chip = label_asset.gt(0).byte().rename('label_chip').clip(area_estudo)

    imagem_base = img_1.addBands(img_2).addBands(label_chip).unmask(0).float()
    print(f'⚙️ MODO_TESTE_RAPIDO={MODO_TESTE_RAPIDO} | positivas={AMOSTRAS_POSITIVAS} | negativas={AMOSTRAS_NEGATIVAS} | tileScale={EXPORT_TILE_SCALE}')
    # Importante: usamos uma banda temporária ('classe_amostra') para a amostragem.
    # Se a propriedade do ponto também se chamar 'label_chip', ela sobrescreve o patch 129x129
    # exportado por sampleRegions e causa erro de reshape no TensorFlow.
    classe_amostra = label_chip.rename('classe_amostra')
    pontos = classe_amostra.stratifiedSample(
        numPoints=AMOSTRAS_POSITIVAS + AMOSTRAS_NEGATIVAS,
        classBand='classe_amostra',
        region=area_estudo,
        scale=SCALE,
        classValues=[0, 1],
        classPoints=[AMOSTRAS_NEGATIVAS, AMOSTRAS_POSITIVAS],
        seed=SEED,
        geometries=True,
        tileScale=EXPORT_TILE_SCALE,
    )

    kernel = ee.Kernel.square(radius=KERNEL_RADIUS, units='pixels', normalize=False)
    pontos_com_shard = pontos.randomColumn('shard_random', seed=SEED)

    tasks = []
    print(f'🧩 Dividindo exportação em {NUM_PARTES_EXPORT} partes para evitar limite de memória do GEE.')
    for parte in range(NUM_PARTES_EXPORT):
        inicio = parte / NUM_PARTES_EXPORT
        fim = (parte + 1) / NUM_PARTES_EXPORT
        pontos_parte = pontos_com_shard.filter(
            ee.Filter.And(
                ee.Filter.gte('shard_random', inicio),
                ee.Filter.lt('shard_random', fim),
            )
        )
        patches = imagem_base.neighborhoodToArray(kernel).sampleRegions(
            collection=pontos_parte,
            scale=SCALE,
            properties=[],
            geometries=False,
            tileScale=EXPORT_TILE_SCALE,
        )
        sufixo = f'PARTE_{parte + 1:02d}_DE_{NUM_PARTES_EXPORT:02d}'
        nome_parte = f'{NOME_EXPORTACAO}_{sufixo}'
        task = ee.batch.Export.table.toDrive(
            collection=patches,
            description=nome_parte,
            folder=DRIVE_FOLDER,
            fileNamePrefix=nome_parte,
            fileFormat='TFRecord',
            selectors=['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2', 'label_chip'],
        )
        task.start()
        tasks.append(task)
        print(f'✅ Tarefa iniciada: {nome_parte} | Task ID: {task.id}')

    return tasks

def aguardar_exportacao_ee(tasks):
    if not isinstance(tasks, (list, tuple)):
        tasks = [tasks]

    prazo_seg = TEMPO_MAX_ESPERA_EXPORTACAO_MIN * 60
    inicio = time.time()

    while True:
        status_list = [task.status() for task in tasks]
        estados = [status.get('state', 'UNKNOWN') for status in status_list]
        resumo = {estado: estados.count(estado) for estado in sorted(set(estados))}
        print(f'⏱️ Status das exportações: {resumo}')

        if all(estado == 'COMPLETED' for estado in estados):
            print('✅ Todas as partes da exportação foram concluídas no Earth Engine.')
            return True

        estados_falha = {'FAILED', 'CANCELLED', 'CANCEL_REQUESTED'}
        if any(estado in estados_falha for estado in estados):
            for status in status_list:
                if status.get('state') in estados_falha:
                    print(f'❌ Parte da exportação não concluída. Status completo: {status}')
            return False

        if time.time() - inicio > prazo_seg:
            print('⏳ Tempo máximo de espera atingido. As exportações podem continuar rodando no painel Tasks.')
            return False

        print(f'⏳ Aguardando {INTERVALO_CHECAGEM_EXPORTACAO_SEG} segundos antes da próxima checagem...')
        time.sleep(INTERVALO_CHECAGEM_EXPORTACAO_SEG)

def aguardar_tfrecord_no_drive():
    # Depois que a task conclui, o Drive pode levar alguns segundos para mostrar o arquivo.
    for tentativa in range(1, 11):
        candidatos = buscar_tfrecords_safrinha()
        if candidatos:
            print('✅ TFRecord apareceu no Drive após a exportação:')
            for i, arquivo in enumerate(candidatos[:10], start=1):
                print(f'  {i}. {arquivo}')
            return candidatos
        print(f'🔎 Aguardando o arquivo aparecer no Drive... tentativa {tentativa}/10')
        time.sleep(15)
    return None

# O treinamento deve usar amostras criadas para a safrinha, não amostras genéricas da safra.
def eh_tfrecord_safrinha_2025(caminho):
    nome = os.path.basename(caminho).lower().replace(' ', '_')
    eh_tfrecord = 'tfrecord' in nome or nome.endswith(('.record', '.record.gz', '.records', '.records.gz'))
    eh_safrinha = 'safrinha' in nome or ('segunda' in nome and 'safra' in nome)
    eh_2025 = '2025' in nome
    eh_versao_atual = VERSAO_AMOSTRAS.lower() in nome
    return eh_tfrecord and eh_safrinha and eh_2025 and (eh_versao_atual or not USAR_APENAS_VERSAO_ATUAL)

def buscar_tfrecords_safrinha():
    encontrados = []
    for raiz, _, nomes in os.walk(pasta_base):
        for nome in nomes:
            caminho = os.path.join(raiz, nome)
            if eh_tfrecord_safrinha_2025(caminho):
                encontrados.append(caminho)
    encontrados = list(dict.fromkeys(encontrados))
    encontrados.sort(key=os.path.getmtime, reverse=True)
    return encontrados

def localizar_tfrecord_safrinha():
    if CAMINHO_TFRECORD_SAFRINHA:
        if os.path.exists(CAMINHO_TFRECORD_SAFRINHA):
            print('✅ Usando TFRecord informado manualmente em CAMINHO_TFRECORD_SAFRINHA.')
            return [CAMINHO_TFRECORD_SAFRINHA]
        raise FileNotFoundError(f'O caminho informado manualmente não existe: {CAMINHO_TFRECORD_SAFRINHA}')

    candidatos = buscar_tfrecords_safrinha()
    if candidatos:
        print('✅ TFRecords específicos da safrinha 2025 encontrados:')
        for i, arquivo in enumerate(candidatos[:10], start=1):
            print(f'  {i}. {arquivo}')
        return candidatos

    if AUTO_CRIAR_TFRECORD_SE_AUSENTE:
        print('⚠️ Nenhum TFRecord específico da safrinha 2025 foi encontrado.')
        print('📦 Iniciando agora a criação/exportação das amostras da safrinha 2025 no Earth Engine...')
        tasks = iniciar_exportacao_tfrecord_safrinha()
        print('✅ Exportação iniciada com sucesso.')
        for i, task in enumerate(tasks, start=1):
            print(f'🆔 Task {i:02d}: {task.id}')

        if AGUARDAR_EXPORTACAO_E_TREINAR:
            print('⏳ Vou aguardar a exportação finalizar para tentar continuar o treino automaticamente nesta execução.')
            if aguardar_exportacao_ee(tasks):
                caminho_exportado = aguardar_tfrecord_no_drive()
                if caminho_exportado:
                    return caminho_exportado
                print('⚠️ A exportação terminou, mas o TFRecord ainda não apareceu no Drive.')
            print('🛑 Não foi possível continuar automaticamente agora. Rode esta célula novamente quando o arquivo aparecer no Drive.')
            return None

        print('⏳ Aguarde a tarefa terminar no painel Tasks/Drive e rode esta célula novamente para treinar.')
        return None

    raise FileNotFoundError(
        'Nenhum TFRecord específico da safrinha 2025 foi encontrado. '
        'Execute primeiro o Passo 1 para criar AMOSTRAS_SAFRINHA_2025_*.tfrecord.gz '
        'ou preencha CAMINHO_TFRECORD_SAFRINHA com o caminho completo do TFRecord da safrinha.'
    )

caminho_arquivo = localizar_tfrecord_safrinha()

if caminho_arquivo is None:
    print('🛑 Treinamento pausado: o TFRecord ainda não está disponível no Drive.')
    print('✅ Se a exportação ainda estiver rodando ou o arquivo demorar a aparecer, execute esta célula novamente depois.')
else:
    caminhos_arquivo = caminho_arquivo if isinstance(caminho_arquivo, list) else [caminho_arquivo]
    compression_type = 'GZIP' if all(caminho.endswith('.gz') for caminho in caminhos_arquivo) else ''
    print(f"📂 Lendo {len(caminhos_arquivo)} arquivo(s) da {NOME_CENARIO}:")
    for caminho in caminhos_arquivo[:20]:
        print(f"  - {caminho}")
    print(f"🗜️ Compressão detectada: {compression_type or 'sem compressão'}")

    # Parâmetros
    KERNEL_SIZE = 128
    READ_SIZE = 129
    BATCH_SIZE = 32
    EPOCHS = 40
    INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
    LABEL_BAND = 'label_chip'

    # Nomes isolados para não sobrescrever os modelos/checkpoints anteriores.
    checkpoint_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Jussara_{NOME_CENARIO}.keras')
    final_path = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{NOME_CENARIO}_FINAL.keras')

    # --- PIPELINE DE DADOS ---
    def parse_and_process(example_proto):
        features_dict = {
            band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
        }
        parsed = tf.io.parse_single_example(example_proto, features_dict)

        inputs_list = []
        for band in INPUT_BANDS:
            dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
            tf.debugging.assert_equal(
                tf.size(dense),
                READ_SIZE * READ_SIZE,
                message=f'Banda {band} não tem patch {READ_SIZE}x{READ_SIZE}. Recrie o TFRecord PATCHES_V2.'
            )
            img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
            img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
            inputs_list.append(img)

        image_stacked = tf.concat(inputs_list, axis=-1)

        dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
        tf.debugging.assert_equal(
            tf.size(dense_lbl),
            READ_SIZE * READ_SIZE,
            message='label_chip não tem patch 129x129. Isso geralmente indica TFRecord antigo com label escalar; recrie o PATCHES_V2.'
        )
        lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
        lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
        return image_stacked, lbl

    # Contagem rápida para não dar erro de tamanho
    print('🔢 Verificando tamanho do arquivo...')
    raw_dataset = tf.data.TFRecordDataset(caminhos_arquivo, compression_type=compression_type)
    N_REAL = sum(1 for _ in raw_dataset)
    print(f'✅ Total de amostras: {N_REAL}')

    if N_REAL < 2:
        raise ValueError('O TFRecord precisa ter pelo menos 2 amostras para separar treino e validação.')

    N_TRAIN = max(1, int(N_REAL * 0.8))
    N_VAL = N_REAL - N_TRAIN
    if N_VAL == 0:
        N_TRAIN -= 1
        N_VAL = 1

    full_dataset = tf.data.TFRecordDataset(caminhos_arquivo, compression_type=compression_type).map(parse_and_process)

    train_ds = full_dataset.take(N_TRAIN).cache().shuffle(N_TRAIN).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    print(f'🧪 Treino: {N_TRAIN} amostras | Validação: {N_VAL} amostras')

    # --- MODELO U-NET ---
    def build_unet(input_shape):
        inputs = layers.Input(shape=input_shape)

        # Encoder
        c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
        p1 = layers.MaxPooling2D()(c1)
        c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
        p2 = layers.MaxPooling2D()(c2)
        c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
        p3 = layers.MaxPooling2D()(c3)

        # Bottleneck
        c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)

        # Decoder
        u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
        u5 = layers.concatenate([u5, c3])
        c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)

        u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
        u6 = layers.concatenate([u6, c2])
        c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)

        u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
        u7 = layers.concatenate([u7, c1])
        c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)

        outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
        return models.Model(inputs=[inputs], outputs=[outputs])

    model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(INPUT_BANDS)))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # --- 🛡️ SALVAMENTO AUTOMÁTICO (SEGURANÇA) ---
    checkpoint_cb = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=False,
        verbose=1
    )

    print(f'🔥 Iniciando retreinamento da {NOME_CENARIO}...')
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[checkpoint_cb]
    )

    # Salvamento final definitivo
    model.save(final_path)
    print(f'✅ SUCESSO! Modelo final da {NOME_CENARIO} salvo em: {final_path}')


## Passo 3 — Prova visual da safrinha 2025

Execute depois do treinamento para carregar o modelo salvo e visualizar exemplos usando o TFRecord específico da safrinha.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from google.colab import drive

# 1. Montar Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print('--- INICIANDO PROVA REAL DA SAFRINHA 2025 ---')

# 2. Localizar o modelo salvo da safrinha 2025
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
NOME_CENARIO = 'Safrinha_2025'
caminho_modelo = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{NOME_CENARIO}_FINAL.keras')

if os.path.exists(caminho_modelo):
    print(f'✅ Arquivo do modelo encontrado: {caminho_modelo}')
    model = tf.keras.models.load_model(caminho_modelo)
    print('✅ Modelo carregado na memória com sucesso!')
else:
    raise FileNotFoundError(f'O arquivo .keras não foi encontrado: {caminho_modelo}')

# 3. Carregar dados da safrinha 2025 para testar
CAMINHO_TFRECORD_SAFRINHA = None
VERSAO_AMOSTRAS = 'PATCHES_V2'
USAR_APENAS_VERSAO_ATUAL = True
def eh_tfrecord_safrinha_2025(caminho):
    nome = os.path.basename(caminho).lower().replace(' ', '_')
    eh_tfrecord = 'tfrecord' in nome or nome.endswith(('.record', '.record.gz', '.records', '.records.gz'))
    eh_safrinha = 'safrinha' in nome or ('segunda' in nome and 'safra' in nome)
    eh_2025 = '2025' in nome
    eh_versao_atual = VERSAO_AMOSTRAS.lower() in nome
    return eh_tfrecord and eh_safrinha and eh_2025 and (eh_versao_atual or not USAR_APENAS_VERSAO_ATUAL)

def buscar_tfrecords_safrinha():
    encontrados = []
    for raiz, _, nomes in os.walk(pasta_base):
        for nome in nomes:
            caminho = os.path.join(raiz, nome)
            if eh_tfrecord_safrinha_2025(caminho):
                encontrados.append(caminho)
    encontrados = list(dict.fromkeys(encontrados))
    encontrados.sort(key=os.path.getmtime, reverse=True)
    return encontrados

def localizar_tfrecord_safrinha():
    if CAMINHO_TFRECORD_SAFRINHA:
        if os.path.exists(CAMINHO_TFRECORD_SAFRINHA):
            print('✅ Usando TFRecord informado manualmente em CAMINHO_TFRECORD_SAFRINHA.')
            return [CAMINHO_TFRECORD_SAFRINHA]
        raise FileNotFoundError(f'O caminho informado manualmente não existe: {CAMINHO_TFRECORD_SAFRINHA}')

    candidatos = buscar_tfrecords_safrinha()
    if candidatos:
        print('✅ TFRecords específicos da safrinha 2025 encontrados:')
        for i, arquivo in enumerate(candidatos[:10], start=1):
            print(f'  {i}. {arquivo}')
        return candidatos

    raise FileNotFoundError(
        'Nenhum TFRecord específico da safrinha 2025 foi encontrado para validação. '
        'Execute primeiro o Passo 1 para criar AMOSTRAS_SAFRINHA_2025_*.tfrecord.gz '
        'ou preencha CAMINHO_TFRECORD_SAFRINHA com o caminho completo do TFRecord da safrinha.'
    )

caminho_dados = localizar_tfrecord_safrinha()
caminhos_dados = caminho_dados if isinstance(caminho_dados, list) else [caminho_dados]
compression_type = 'GZIP' if all(caminho.endswith('.gz') for caminho in caminhos_dados) else ''
print(f'📂 Validando com {len(caminhos_dados)} arquivo(s):')
for caminho in caminhos_dados[:20]:
    print(f'  - {caminho}')
print(f"🗜️ Compressão detectada: {compression_type or 'sem compressão'}")

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        tf.debugging.assert_equal(
            tf.size(dense),
            READ_SIZE * READ_SIZE,
            message=f'Banda {band} não tem patch {READ_SIZE}x{READ_SIZE}. Recrie o TFRecord PATCHES_V2.'
        )
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    tf.debugging.assert_equal(
        tf.size(dense_lbl),
        READ_SIZE * READ_SIZE,
        message='label_chip não tem patch 129x129. Isso geralmente indica TFRecord antigo com label escalar; recrie o PATCHES_V2.'
    )
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Pega apenas 1 lote de até 10 imagens
dataset = tf.data.TFRecordDataset(caminhos_dados, compression_type=compression_type)
dataset = dataset.map(parse_fast).batch(10).take(1)

# 4. Gerar previsões
print('🔮 Gerando previsões com o modelo carregado...')
imgs, labels = next(iter(dataset))
preds = model.predict(imgs)

# 5. Visualizar
n_exemplos = min(5, imgs.shape[0])
plt.figure(figsize=(15, 3 * n_exemplos))
print('\nLEGENDA: Esquerda=Satélite | Meio=Gabarito | Direita=O que a IA Aprendeu')

for i in range(n_exemplos):
    # Satélite (NDVI do primeiro momento)
    plt.subplot(n_exemplos, 3, i * 3 + 1)
    plt.imshow(imgs[i][:, :, 2], cmap='RdYlGn', vmin=0, vmax=0.8)
    plt.axis('off')
    if i == 0:
        plt.title('Satélite (NDVI)')

    # Gabarito
    plt.subplot(n_exemplos, 3, i * 3 + 2)
    plt.imshow(labels[i][:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0:
        plt.title('Gabarito Real')

    # Predição da IA
    plt.subplot(n_exemplos, 3, i * 3 + 3)
    plt.imshow(preds[i][:, :, 0], cmap='magma', vmin=0, vmax=1)
    plt.axis('off')
    if i == 0:
        plt.title('IA Safrinha 2025')

plt.tight_layout()
plt.show()
